# 04 — Q-Learning (Orang 2)

Agen dilatih di `engine/rl_env.py` milik Orang 3, memakai `predictions.csv` (kelompok B) dari notebook 03. Hasilnya tiga Q-table di `models/v1/` (rasio biaya 50, 100, 200 : 10), `kurva_reward.json`, lalu `evaluasi.json` disusun ulang dengan `engine.evaluate` milik Orang 3 supaya baris **Agen Q-Learning** dan tabel policy ikut terisi.

Masalahnya *optimal stopping*: setiap siklus agen memilih lanjut (+1) atau rawat (−10, selesai); lanjut di siklus terakhir berarti rusak (−biaya gagal). State = bin `rul_pred` × tahap kesehatan = 18, dengan rumus di `rl_config.json`.

In [ ]:
# Colab: jalankan sel ini dulu (repo, branch, dan data mentah).
import os, shutil, sys
from google.colab import drive
drive.mount('/content/drive')
PATH_DRIVE = '/content/drive/MyDrive/Praktikum MLN/Dataset ML'
BRANCH = 'model'
if not os.path.exists('/content/prognosense'):
    !git clone https://github.com/Arsha8Anuga/PrognoSense /content/prognosense
%cd /content/prognosense
!git checkout {BRANCH} 2>/dev/null || git checkout -b {BRANCH}
sys.path.insert(0, '/content/prognosense')

## 1. Data dan konfigurasi

Agen **hanya** dilatih dengan mesin kelompok B. Kelompok C disimpan untuk evaluasi, dan A sudah dipakai melatih model RUL.

In [ ]:
import json, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from engine.rl_env import MaintenanceEnv
from engine import qlearning as QL, evaluate as EV

HASIL, MODEL = 'data/results', 'models/v1'
preds = pd.read_csv(f'{HASIL}/predictions.csv')
unit_B = preds.loc[preds['kelompok'] == 'B', 'unit'].unique()
rl_config = json.load(open(f'{MODEL}/rl_config.json'))
print('mesin B untuk latihan:', len(unit_B), '| state:', rl_config['n_state'], '| rumus:', rl_config['rumus_state'])

## 2. Uji kewarasan (Kontrak 4)

Sebelum memakai RUL prediksi, agen dilatih dengan RUL **sebenarnya**. Kalau dengan informasi sempurna pun agen masih sering membiarkan mesin rusak, yang salah adalah lingkungannya atau rewardnya, bukan model RUL. Pada jalan acuan: 0% gagal, rata-rata 9 siklus terbuang.

In [ ]:
uji = preds.assign(rul_pred=preds['rul_aktual'])            # agen diberi RUL sebenarnya
q_uji, _ = QL.latih(MaintenanceEnv(uji, unit_B, bins=rl_config['batas_bin'], biaya_gagal=100))
cek = {h['nama']: h for h in EV.bandingkan(uji, 100, qtable=q_uji, bins=rl_config['batas_bin'])}['Agen Q-Learning']
print('uji kewarasan: gagal', cek['gagal_pct'], '% | siklus terbuang', cek['siklus_terbuang'])
assert cek['gagal_pct'] <= 5, 'dengan RUL sebenarnya agen masih sering gagal: lingkungannya yang salah'

## 3. Melatih tiga Q-table

Parameter dari Kontrak 4: α = 0,1, γ = 0,99, ε turun eksponensial dari 1,0 ke 0,01 (`decay = 0.001` per episode), 5.000 episode, mesin diacak dari B.

- **α (learning rate):** seberapa besar Q-value digeser oleh setiap pengalaman baru.
- **γ (discount):** seberapa penting reward di masa depan. 0,99 berarti agen "sabar": ia menghargai ratusan siklus produktif yang akan datang.
- **ε (eksplorasi):** peluang memilih aksi acak. Di awal agen mencoba-coba; di akhir ia hampir selalu memakai aksi terbaiknya.

Rumus pembaruan yang dijalankan setiap langkah:

`Q[s,a] ← Q[s,a] + α · (r + γ · max Q[s′] − Q[s,a])`

Kurva reward yang naik lalu mendatar menandakan agen sudah berhenti menemukan cara yang lebih baik.

In [ ]:
kurva = {}
for r in EV.RASIO:
    env = MaintenanceEnv(preds, unit_B, bins=rl_config['batas_bin'], biaya_rawat=10, biaya_gagal=r)
    q, reward_ep = QL.latih(env)
    np.save(f'{MODEL}/qtable_r{r}.npy', q)
    kurva[r] = QL.kurva(reward_ep)
    print(f'rasio {r}: reward rata-rata 100 episode terakhir = {kurva[r][-1]["rata2"]}')
json.dump(kurva[100], open(f'{HASIL}/kurva_reward.json', 'w'))

fig, ax = plt.subplots(figsize=(7, 3))
for r in EV.RASIO:
    ax.plot([k['episode'] for k in kurva[r]], [k['rata2'] for k in kurva[r]], label=f'rasio {r}:10')
ax.set(xlabel='episode', ylabel='reward rata-rata per 100 episode', title='Kurva belajar Q-Learning'); ax.legend(); plt.show()

## 4. Menyusun `evaluasi.json`

`engine.evaluate.tulis_evaluasi` (Orang 3) memilih parameter strategi pembanding di B dan menguji semua strategi di C.

**Cara membaca tabel.** Biaya hanya menghitung 10 per perawatan dan biaya gagal per mesin rusak. Siklus produktif yang terbuang tidak dihitung, sehingga strategi mana pun yang tidak pernah gagal berbiaya sama dengan oracle (1.020). Karena itu bandingkan juga **tingkat gagal** dan **siklus terbuang**.

Pada jalan acuan (rasio 100): agen membuang rata-rata 11,8 siklus, jauh lebih sedikit dari aturan ambang (19,4), tetapi membiarkan 1 dari 102 mesin rusak (0,98%), sedangkan ambang tidak pernah gagal. Agen lebih berani, ambang lebih hati-hati. Pada rasio 50 agen lebih berani lagi (4,9% gagal). Rasio 200 menghasilkan kebijakan yang sama dengan 100, karena 6 bin RUL terlalu kasar untuk membuat agen lebih hati-hati. Ini dilaporkan apa adanya.

In [ ]:
EV.tulis_evaluasi(HASIL, MODEL)
ev = json.load(open(f'{HASIL}/evaluasi.json'))
tabel = pd.DataFrame(ev['rl']['strategi'])
tabel[['rasio', 'nama', 'parameter', 'gagal_pct', 'siklus_terbuang', 'siklus_produktif', 'biaya']]

## 5. Tabel policy (rasio 100)

18 baris: untuk setiap kombinasi bin RUL dan tahap, aksi yang dipilih agen beserta Q-value-nya. State yang jarang dikunjungi bisa punya Q mendekati 0; artinya agen belum cukup belajar di sana, bukan bahwa kedua aksi sama baiknya.

In [ ]:
pd.DataFrame(ev['rl']['policy'])